# M4 · Visión como *input* — launcher

Este notebook **solo orquesta** el módulo `M4/vision/`. No reimplementa la visión, el harness ni la app.

- **Qué hace:** lee las imágenes del eval set de M3 desde Google Drive (`M4/data/imagenes/`) y genera **2 JSON** de transcripción: uno para el caso normal (`limpio`) y otro para el caso realista/reto (`foto_realista`).
- **No consume API de Groq:** el VLM (`Qwen/Qwen2-VL-2B-Instruct`) corre localmente en la GPU de Colab.


## 1 · Clonar repositorio e instalar dependencias

In [9]:
RAMA = "LuisM4"

!git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo
%cd /content/repo
!git pull
!pip install -r M4/vision/requirements.txt -q

fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo
Already up to date.


## 2 · Montar Drive y definir `PROJECT_ROOT`

Lee las imágenes de `<PROJECT_ROOT>/M4/data/imagenes/` y guarda los 2 JSON en `<PROJECT_ROOT>/M4/vision/predictions/`.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

with open("/content/repo/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

print("PROJECT_ROOT:", PROJECT_ROOT)
for nivel in ("limpio", "foto_realista"):
    csv_path = PROJECT_ROOT / f"M4/data/imagenes/casos/{nivel}.csv"
    print(f"  CSV {nivel}:", csv_path.exists(), "->", csv_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
  CSV limpio: True -> /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/casos/limpio.csv
  CSV foto_realista: True -> /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/casos/foto_realista.csv


## 3 · Generar los 2 JSON: Caso Normal (`limpio`) vs. Caso Realista/Reto (`foto_realista`)

- Si `LIMITE = 1`, solo transcribe el primer ejemplo (`caso_01_p1`) en cada JSON (prueba ultrarrápida).
- Si `LIMITE = None`, transcribe los 15 casos (`caso_01` a `caso_15`) del eval set de M3.
- Si quieres que también genere el cache NER (`M2/predictions/vision_vlm_*.json`), pon `CORRER_NER = True`.

In [11]:
!pip install "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
LIMITE = None         # Pon None para correr los 15 casos de M3, o 1 para probar solo con caso_01
CORRER_NER = True  # Pon True si tienes el adaptador de M1 en Drive y quieres el cache NER

flag_limite = f"--limite {LIMITE}" if LIMITE else ""
flag_ner = "--ner" if CORRER_NER else ""

# 1) JSON Normal (limpio)
!python M4/vision/run_vision.py --config M4/vision/config.yaml \
    --backend vlm \
    --manifest M4/data/imagenes/casos/limpio.csv \
    --nivel limpio {flag_limite} {flag_ner}

# 2) JSON Realista / Reto (foto_realista)
!python M4/vision/run_vision.py --config M4/vision/config.yaml \
    --backend vlm \
    --manifest M4/data/imagenes/casos/foto_realista.csv \
    --nivel foto_realista {flag_limite} {flag_ner}

[02:01:16] Casos: 15 | backends: vlm | nivel: limpio | grupo: todos | raíz: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
[02:01:16] Transcribiendo con backend=vlm ...
Loading weights: 100% 729/729 [00:01<00:00, 633.58it/s]
[02:01:35]   [1/15] caso_01__p1 -> ok (17.934s, vlm)
[02:01:39]   [2/15] caso_02__p1 -> ok (4.313s, vlm)
[02:01:43]   [3/15] caso_03__p1 -> ok (4.317s, vlm)
[02:01:47]   [4/15] caso_04__p1 -> ok (4.055s, vlm)
[02:01:51]   [5/15] caso_05__p1 -> ok (3.404s, vlm)
[02:01:55]   [6/15] caso_06__p1 -> ok (4.367s, vlm)
[02:01:59]   [7/15] caso_07__p1 -> ok (3.462s, vlm)
[02:02:02]   [8/15] caso_08__p1 -> ok (2.982s, vlm)
[02:02:05]   [9/15] caso_09__p1 -> ok (2.962s, vlm)
[02:02:08]   [10/15] caso_10__p1 -> ok (3.357s, vlm)
[02:02:12]   [11/15] caso_11__p1 -> ok (3.595s, vlm)
[02:02:16]   [12/15] caso_12__p1 -> ok (3.311s, vlm)
[02:02:19]   [13/15] caso_13__p1 -> ok (3.081s, vlm)
[02:02:22]   [14/15] caso_14__p1 -> ok (3.1s, vlm)
[02:02:25]   [15/15] caso_15__p1 -> ok (3.

## 4 · Verificar los 2 JSON generados en Drive

In [13]:
for nombre in ("transcripciones_vlm_limpio.json", "transcripciones_vlm_foto_realista.json"):
    ruta = PROJECT_ROOT / "M4/vision/predictions" / nombre
    print("=" * 70)
    print(nombre, "| existe:", ruta.exists())
    if ruta.exists():
        datos = json.loads(ruta.read_text(encoding="utf-8"))
        print("Casos incluidos:", list(datos.keys()))
        primero = next(iter(datos.values()))
        print("Ejemplo:", json.dumps(primero, ensure_ascii=False, indent=2)[:800])

transcripciones_vlm_limpio.json | existe: True
Casos incluidos: ['caso_01__p1', 'caso_02__p1', 'caso_03__p1', 'caso_04__p1', 'caso_05__p1', 'caso_06__p1', 'caso_07__p1', 'caso_08__p1', 'caso_09__p1', 'caso_10__p1', 'caso_11__p1', 'caso_12__p1', 'caso_13__p1', 'caso_14__p1', 'caso_15__p1']
Ejemplo: {
  "doc_id": "caso_01__p1",
  "texto": "Varón de 52 años, con sobrepeso (IMC 29.3 kg/m2) y vida sedentaria, que acude a consulta de medicina preventiva. En analíticas de control rutinario presenta glucemia plasmática en ayunas de 142 mg/dL y 138 mg/dL en dos tomas separadas, con una hemoglobina glicosilada (HbA1c) de 7.2%. No refiere antecedentes de enfermedad cardiovascular ni renal, y se encuentra asintomático, confirmándose el diagnóstico de diabetes mellitus tipo 2 de reciente comienzo.",
  "backend": "vlm",
  "segundos": 17.934,
  "valvula": false,
  "error": null,
  "imagen": "/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/limpio/caso_01_p1.png",
  "sha256": "db1adcd1